In [4]:
from qm import QuantumMachinesManager
import time
import matplotlib.pyplot as plt
from configuration import *
import numpy as np
import scipy.optimize as opti
from auto_mixer_tools_visa import KeysightXSeries
import pyvisa as visa
from qm import SimulationConfig
from qm.qua import *
from qm import LoopbackInterface
from helper_functions import *


qmm = QuantumMachinesManager()
qm = qmm.open_qm(config)

address = "TCPIP0::192.168.0.102::inst0::INSTR"
calib = KeysightXSeries(address, qm)

2022-06-09 18:45:48,545 - qm - INFO - Performing health check
2022-06-09 18:45:48,551 - qm - INFO - Health check passed
2022-06-09 18:45:48,582 - qm - INFO - Flags: 
2022-06-09 18:45:48,582 - qm - INFO - Sending program to QOP
2022-06-09 18:45:48,594 - qm - INFO - Executing program


In [5]:
with program() as mixer_calibration_pulse:
    with infinite_loop_():
        
        play("const"*amp(0.0), "qubit")
        play("const"*amp(0.0), "rr")
        
job = qm.execute(mixer_calibration_pulse)

2022-06-09 18:45:49,639 - qm - INFO - Flags: 
2022-06-09 18:45:49,639 - qm - INFO - Sending program to QOP
2022-06-09 18:45:49,653 - qm - INFO - Executing program


In [ ]:
qe_freq = {"qubit": [qubit_LO, [1,2]],
            "rr": [rr_LO, [9,10]]
          }
qe = "qubit"
center_freq, qe_ch = qe_freq[qe] 
span = 50
calib.set_bandwidth(5)
calib.set_sweep_points(501)

calib.set_center_freq(center_freq)
calib.set_span(span)
calib.active_marker(1)
calib.set_marker_freq(1, center_freq)

def set_dc_offset_get_leakage(offset_arr, qe, verbose=False):
    
    I_offset, Q_offset = offset_arr[0], offset_arr[1]
    
    qm.set_output_dc_offset_by_element(qe, "I", I_offset)
    qm.set_output_dc_offset_by_element(qe, "Q", Q_offset)
    
    time.sleep(1)
    
    leakage = calib.query_marker(1)
    if verbose:
        print("Current leakage is {0} dBm".format(leakage))
    
    if leakage < -95:
         return -300
        
    return leakage

In [ ]:
from scipy.optimize import minimize

init_vals = [0.0, 0.0]
#lower_bound = -60
bnds = ((-0.3, 0.3), (-0.3, 0.3))

xatol = 1e-4  # 1e-4 change in DC offset or gain/phase
#fatol = 3  # dB change tolerance
maxiter = 50  # 50 iterations should be more then enough, but can be changed.
initial_simplex = np.zeros([3, 2])
# initial_simplex[0, :] = [0, 0]
# initial_simplex[1, :] = [0, -0.2]
# initial_simplex[2, :] = [-0.1, 0]

initial_simplex[0, :] = [0, -0.2]
initial_simplex[1, :] = [0.2, 0.2]
initial_simplex[2, :] = [-0.2, 0.2]

res = minimize(set_dc_offset_get_leakage, 
               init_vals, 
               args=qe, 
               method="nelder-mead", 
               bounds=bnds, 
               options={
                "xatol": xatol,
#                "fatol": fatol,
                "initial_simplex": initial_simplex,
                "maxiter": maxiter,
               }
              )
print("Final leakage is {0} dBm".format(calib.query_marker(1)))
print(f"Calibrated Mixer offsets for {qe} are {res.x}")

In [ ]:
def update_config_offsetvals(file,channels,offsetvalues,par='analog_outputs'):
    '''
    Used to update the offset values in the configuration file.
    Input example ('configuration.py',[2,4,9],[-0.123,1.2,-0.035],par='analog_inputs')
    '''
    def line_num_for_phrase_in_file(phrase, filename,linelimit):
        with open(filename,'r') as f:
            lines = f.readlines()
            for i in range(linelimit[0],linelimit[1]+1):
                if phrase in lines[i]:
                    return i
        return -1  

    ofile = open(file,'r')
    lines = ofile.readlines()
    linenum = line_num_for_phrase_in_file(par,file,(0,len(lines)))

    for i,ch_no in enumerate(channels):
        t_linenu = line_num_for_phrase_in_file(f'{ch_no}: ',file,(linenum+1,linenum+2+len(channels)))
        if t_linenu == -1:
            print(f'Channel {ch_no} not present in config file. Not updated.')
        else:
            oldoffsetval = lines[t_linenu].split(':')[2].split('}')[0]
            lines[t_linenu] = lines[t_linenu].replace(oldoffsetval,str(' '+ str(offsetvalues[i])))

    ofile = open(file,'w')
    ofile.writelines(lines)
    ofile.close()
    return print(f'{par} offset values updated for channels {channels}.')

file='configuration.py'
update_config_offsetvals(file,qe_ch,res.x,par='analog_outputs')